<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/WithoutKD_Al9EE_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip "/content/Apple.zip"

Streaming output truncated to the last 5000 lines.
  inflating: Apple/Scab/Scab (1368).jpg  
  inflating: Apple/Scab/Scab (1369).jpg  
  inflating: Apple/Scab/Scab (137).jpg  
  inflating: Apple/Scab/Scab (1370).jpg  
  inflating: Apple/Scab/Scab (1371).jpg  
  inflating: Apple/Scab/Scab (1372).jpg  
  inflating: Apple/Scab/Scab (1373).jpg  
  inflating: Apple/Scab/Scab (1374).jpg  
  inflating: Apple/Scab/Scab (1375).jpg  
  inflating: Apple/Scab/Scab (1376).jpg  
  inflating: Apple/Scab/Scab (1377).jpg  
  inflating: Apple/Scab/Scab (1378).jpg  
  inflating: Apple/Scab/Scab (1379).jpg  
  inflating: Apple/Scab/Scab (138).jpg  
  inflating: Apple/Scab/Scab (1380).jpg  
  inflating: Apple/Scab/Scab (1381).jpg  
  inflating: Apple/Scab/Scab (1382).jpg  
  inflating: Apple/Scab/Scab (1383).jpg  
  inflating: Apple/Scab/Scab (1384).jpg  
  inflating: Apple/Scab/Scab (1385).jpg  
  inflating: Apple/Scab/Scab (1386).jpg  
  inflating: Apple/Scab/Scab (1387).jpg  
  inflating: Apple/Scab/Sca

In [ ]:
!pip install split-folders

In [ ]:
import splitfolders

splitfolders.ratio(
    "/content/Apple",
    output="/content/Apple_split",
    seed=42,
    ratio=(0.7, 0.15, 0.15)
)

Copying files: 11258 files [00:06, 1768.03 files/s]


In [ ]:
DATA_DIR = "/content/Apple_split"
IMAGE_SIZE = (224, 224)   # ResNet50 input size
BATCH_SIZE = 16
NUM_CLASSES = 3
EPOCHS = 10

In [ ]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR + "/train",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = val_test_datagen.flow_from_directory(
    DATA_DIR + "/val",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_gen = val_test_datagen.flow_from_directory(
    DATA_DIR + "/test",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 7879 images belonging to 3 classes.
Found 1687 images belonging to 3 classes.
Found 1692 images belonging to 3 classes.


In [ ]:
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
)

base_model.trainable = False  # Freeze for transfer learning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,593,859 (90.00 MB)

 Trainable params: 6,147 (24.01 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
callbacks = [
    EarlyStopping(patience=6, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.2, patience=3, verbose=1),
    ModelCheckpoint("best_resnet50.h5", save_best_only=True)
]

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - accuracy: 0.5135 - loss: 1.1333

493/493 ━━━━━━━━━━━━━━━━━━━━ 167s 314ms/step - accuracy: 0.5137 - loss: 1.1329 - val_accuracy: 0.8672 - val_loss: 0.4800 - learning_rate: 1.0000e-04
Epoch 2/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.7630 - loss: 0.5876

493/493 ━━━━━━━━━━━━━━━━━━━━ 140s 284ms/step - accuracy: 0.7630 - loss: 0.5875 - val_accuracy: 0.8933 - val_loss: 0.3684 - learning_rate: 1.0000e-04
Epoch 3/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.8375 - loss: 0.4329

493/493 ━━━━━━━━━━━━━━━━━━━━ 140s 283ms/step - accuracy: 0.8375 - loss: 0.4329 - val_accuracy: 0.9016 - val_loss: 0.3194 - learning_rate: 1.0000e-04
Epoch 4/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.8637 - loss: 0.3682

...

493/493 ━━━━━━━━━━━━━━━━━━━━ 143s 291ms/step - accuracy: 0.8865 - loss: 0.3125 - val_accuracy: 0.9105 - val_loss: 0.2702 - learning_rate: 1.0000e-04
Epoch 7/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step - accuracy: 0.9008 - loss: 0.2766

493/493 ━━━━━━━━━━━━━━━━━━━━ 137s 279ms/step - accuracy: 0.9008 - loss: 0.2766 - val_accuracy: 0.9105 - val_loss: 0.2660 - learning_rate: 1.0000e-04
Epoch 8/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 258ms/step - accuracy: 0.9020 - loss: 0.2768

493/493 ━━━━━━━━━━━━━━━━━━━━ 138s 280ms/step - accuracy: 0.9020 - loss: 0.2768 - val_accuracy: 0.9200 - val_loss: 0.2411 - learning_rate: 1.0000e-04
Epoch 9/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - accuracy: 0.9127 - loss: 0.2566

493/493 ━━━━━━━━━━━━━━━━━━━━ 139s 281ms/step - accuracy: 0.9127 - loss: 0.2567 - val_accuracy: 0.9188 - val_loss: 0.2368 - learning_rate: 1.0000e-04
Epoch 10/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.9043 - loss: 0.2651

493/493 ━━━━━━━━━━━━━━━━━━━━ 140s 283ms/step - accuracy: 0.9043 - loss: 0.2651 - val_accuracy: 0.9206 - val_loss: 0.2316 - learning_rate: 1.0000e-04


In [ ]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 357ms/step - accuracy: 0.8615 - loss: 0.4109

493/493 ━━━━━━━━━━━━━━━━━━━━ 248s 392ms/step - accuracy: 0.8616 - loss: 0.4106 - val_accuracy: 0.9656 - val_loss: 0.1163 - learning_rate: 1.0000e-05
Epoch 2/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step - accuracy: 0.9477 - loss: 0.1545

493/493 ━━━━━━━━━━━━━━━━━━━━ 166s 337ms/step - accuracy: 0.9477 - loss: 0.1545 - val_accuracy: 0.9751 - val_loss: 0.0801 - learning_rate: 1.0000e-05
Epoch 3/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step - accuracy: 0.9678 - loss: 0.0982

493/493 ━━━━━━━━━━━━━━━━━━━━ 165s 335ms/step - accuracy: 0.9678 - loss: 0.0982 - val_accuracy: 0.9763 - val_loss: 0.0676 - learning_rate: 1.0000e-05
Epoch 4/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 163s 331ms/step - accuracy: 0.9716 - loss: 0.0807 - val_accuracy: 0.9739 - val_loss: 0.0731 - learning_rate: 1.0000e-05
Epoch 5/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 162s 329ms/step - accuracy: 0.9720 - loss: 0.0850 - val_accuracy: 0.9793 - val_loss: 0.0698 - learning_rate: 1.0000e-05
Epoch 6/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.9815 - loss: 0.0520

493/493 ━━━━━━━━━━━━━━━━━━━━ 168s 341ms/step - accuracy: 0.9815 - loss: 0.0520 - val_accuracy: 0.9769 - val_loss: 0.0566 - learning_rate: 1.0000e-05
Epoch 7/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.9868 - loss: 0.0371

493/493 ━━━━━━━━━━━━━━━━━━━━ 168s 340ms/step - accuracy: 0.9867 - loss: 0.0371 - val_accuracy: 0.9804 - val_loss: 0.0506 - learning_rate: 1.0000e-05
Epoch 8/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.9860 - loss: 0.0400

493/493 ━━━━━━━━━━━━━━━━━━━━ 170s 345ms/step - accuracy: 0.9860 - loss: 0.0400 - val_accuracy: 0.9822 - val_loss: 0.0495 - learning_rate: 1.0000e-05
Epoch 9/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 165s 334ms/step - accuracy: 0.9847 - loss: 0.0443 - val_accuracy: 0.9804 - val_loss: 0.0584 - learning_rate: 1.0000e-05
Epoch 10/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 164s 332ms/step - accuracy: 0.9866 - loss: 0.0357 - val_accuracy: 0.9804 - val_loss: 0.0593 - learning_rate: 1.0000e-05


In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f"Test Accuracy: {test_acc:.4f}")
# Predictions
y_true = test_gen.classes
y_pred_prob = model.predict(test_gen)
y_pred = np.argmax(y_pred_prob, axis=1)

class_labels = list(test_gen.class_indices.keys())

# Classification Report
report = classification_report(
    y_true, y_pred,
    target_names=class_labels,
    digits=4
)

print("Classification Report:\n")
print(report)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# Cohen's Kappa
kappa = cohen_kappa_score(y_true, y_pred)
print(f"\nCohen's Kappa: {kappa:.4f}")

106/106 ━━━━━━━━━━━━━━━━━━━━ 14s 134ms/step - accuracy: 0.9860 - loss: 0.0505
Test Accuracy: 0.9852
106/106 ━━━━━━━━━━━━━━━━━━━━ 18s 133ms/step
Classification Report:

              precision    recall  f1-score   support

      Health     0.9892    0.9892    0.9892       465
        Rust     0.9829    0.9710    0.9769       414
        Scab     0.9841    0.9902    0.9871       813

    accuracy                         0.9852      1692
   macro avg     0.9854    0.9835    0.9844      1692
weighted avg     0.9852    0.9852    0.9852      1692

Confusion Matrix:
[[460   3   2]
 [  1 402  11]
 [  4   4 805]]

Cohen's Kappa: 0.9767


In [ ]:
model.save("apple_resnet50_final.h5")

model_size = os.path.getsize("apple_resnet50_final.h5") / (1024 * 1024)
print(f"\nModel Size: {model_size:.2f} MB")


Model Size: 270.36 MB


In [ ]:
from google.colab import files
files.download("apple_resnet50_final.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>